# Future Sales Prediction

**Tabular Regression Project** — Predict future sales from historical data.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Kaggle retail/sales data
Target: Sales amount

## 2 · Project Overview

We predict **future sales** using retail/sales data. Sales forecasting is a core business problem — accurate predictions improve inventory management, staffing, and revenue planning. We treat this as a tabular regression task.

## 3 · Learning Objectives

1. Work with retail/sales datasets.
2. Handle date-based feature engineering.
3. Apply regression models to sales forecasting.
4. Use LazyPredict and FLAML for model selection.
5. Understand business impact of sales predictions.

## 4 · Problem Statement

Predict **future sales amounts** from historical sales records including product, store, and temporal features.

## 5 · Why This Project Matters

- **Sales forecasting** directly impacts supply chain and inventory decisions.
- Overprediction leads to excess stock; underprediction leads to stockouts.
- One of the most common ML use cases in retail and e-commerce.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Kaggle: rohitsahoo/sales-forecasting |
| **Target** | Sales amount |
| **Features** | Ship Mode, Segment, Country, City, State, Category, Sub-Category, etc. |

## 7 · Dataset Source and License Notes

- **Source**: Kaggle Sales Forecasting dataset.
- **License**: Educational use.
- **Note**: Contains US retail sales data with geographic and product category features.

## 8 · Environment Setup

In [1]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('kagglehub')
print('All packages ready.')

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

Imports complete.


## 10 · Configuration / Constants

In [3]:
SEED = 42
TEST_SIZE = 0.2
MAX_ROWS = 50000
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [4]:
import kagglehub, glob

try:
    path = kagglehub.dataset_download('rohitsahoo/sales-forecasting')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download error: {e}')
    path = SAVE_DIR

csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV found in {path}'
df = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0], encoding='latin1')
print(f'Loaded: {df.shape}')
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
print(f'Columns: {list(df.columns)}')
df.head()

Downloaded to: C:\Users\ahmad\.cache\kagglehub\datasets\rohitsahoo\sales-forecasting\versions\2
Loaded: (9800, 18)
Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


## 12 · Data Validation Checks

In [5]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

Missing values:
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64

Duplicates: 0


## 13 · Exploratory Data Analysis

In [6]:
# Find the target column
target_candidates = [c for c in df.columns if any(kw in c.lower() for kw in ['sales', 'revenue', 'amount', 'profit'])]
if target_candidates:
    TARGET = target_candidates[0]
else:
    num_cols = df.select_dtypes(include='number').columns.tolist()
    TARGET = num_cols[-1] if num_cols else df.columns[-1]

print(f'Target: {TARGET}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET].hist(bins=50, ax=axes[0], edgecolor='black')
axes[0].set_title(f'{TARGET} Distribution')
if 'Category' in df.columns:
    df.groupby('Category')[TARGET].mean().sort_values().plot.barh(ax=axes[1])
    axes[1].set_title(f'Avg {TARGET} by Category')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

Target: Sales


## 14 · Target Analysis

In [7]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

count     9800.000000
mean       230.769059
std        626.651875
min          0.444000
25%         17.248000
50%         54.490000
75%        210.605000
max      22638.480000
Name: Sales, dtype: float64

Skewness: 12.98


## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [8]:
from sklearn.preprocessing import LabelEncoder

# Parse date columns
for col in df.columns:
    if 'date' in col.lower() or 'order' in col.lower():
        try:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            if df[col].dtype == 'datetime64[ns]':
                df[f'{col}_year'] = df[col].dt.year
                df[f'{col}_month'] = df[col].dt.month
                df[f'{col}_day'] = df[col].dt.day
                df = df.drop(columns=[col])
        except: pass

# Drop ID/name columns + high cardinality
for col in df.select_dtypes(include='object').columns:
    nuniq = df[col].nunique()
    if nuniq > 100 or nuniq < 2 or 'id' in col.lower() or 'name' in col.lower():
        df = df.drop(columns=[col])
    else:
        df[col] = df[col].fillna('Unknown')
        df[col] = LabelEncoder().fit_transform(df[col])

for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

print(f'Preprocessed: {df.shape}')

Preprocessed: (9800, 18)


## 17 · Feature Engineering

In [9]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Replace inf with NaN, then fill
X = X.replace([np.inf, -np.inf], np.nan)
for col in X.columns:
    X[col] = X[col].fillna(X[col].median() if X[col].median() == X[col].median() else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (7840, 17), Test: (1960, 17)


## 18 · Baseline Model

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

Baseline LR: RMSE=815.68, MAE=300.94, R²=0.0046


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

                               Adjusted R-Squared  R-Squared        RMSE  \
Model                                                                      
GradientBoostingRegressor                0.157157   0.171499  525.803272   
HistGradientBoostingRegressor            0.149948   0.164413  528.047041   
LGBMRegressor                            0.139982   0.154617  531.133357   
MLPRegressor                             0.059541   0.075544  555.417908   
BaggingRegressor                         0.010857   0.027689  569.612318   
PoissonRegressor                         0.000064   0.017080  572.711500   
OrthogonalMatchingPursuitCV             -0.000013   0.017004  572.733716   
SGDRegressor                            -0.000918   0.016114  572.992780   
Lasso                                   -0.001601   0.015443  573.188216   
LassoLars                               -0.001601   0.015443  573.188235   
LinearRegression                        -0.001799   0.015249  573.244719   
Lars        

## 20 · FLAML AutoML Run

In [12]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

FLAML failed: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## 21 · Boosting Models

In [13]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

CatBoost: RMSE=764.05, MAE=251.24, R²=0.1266
LightGBM: RMSE=767.86, MAE=262.33, R²=0.1179


XGBoost: RMSE=796.70, MAE=267.59, R²=0.0504


## 22 · Top 3 Model Selection

In [14]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

All models ranked by RMSE:
  CatBoost              RMSE=764.05  MAE=251.24  R²=0.1266
  LightGBM              RMSE=767.86  MAE=262.33  R²=0.1179
  XGBoost               RMSE=796.70  MAE=267.59  R²=0.0504
  Baseline_LR           RMSE=815.68  MAE=300.94  R²=0.0046

Top 3: ['CatBoost', 'LightGBM', 'XGBoost']


## 23 · Final Eval of Top 3

In [15]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

Top 3 Final Results:
CatBoost: RMSE=764.05, MAE=251.24, R²=0.1266
LightGBM: RMSE=767.86, MAE=262.33, R²=0.1179
XGBoost: RMSE=796.70, MAE=267.59, R²=0.0504


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- **Category** and **Sub-Category** are typically the strongest drivers.
- Geographic features (region, state) show distinct sales patterns.
- Discount and quantity interact to affect total sales.
- These models help retailers with demand planning and inventory allocation.

## 26 · Limitations

- Historical data doesn't capture market shifts or new products.
- No external factors (economy, competition, seasonality beyond dates).
- Sales are aggregated — individual transaction prediction would differ.

## 27 · How to Improve

1. Add holiday/event features.
2. Include economic indicators.
3. Add product-level time-series lags.
4. External data: weather, promotions, competitor pricing.

## 28 · Production Considerations

- Regular retraining as product mix changes.
- Integration with inventory management systems.
- Batch vs real-time prediction pipeline.
- Monitoring for concept drift in sales patterns.

## 29 · Common Mistakes

1. Including Order ID as a feature.
2. Not parsing date columns properly.
3. Ignoring the heavy right skew in sales.
4. Confusing sales prediction with time-series forecasting.

## 30 · Mini Challenge

1. Try log-transforming the target.
2. Add interaction features (Category × Region).
3. Build separate models per product category.
4. Add rolling average features if dates are available.

## 31 · Final Summary

- Sales prediction is a high-impact retail ML problem.
- Feature engineering from categorical and date features matters significantly.
- Gradient-boosting models handle the mixed feature types well.

In [17]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')

Metrics saved to metrics.json

Notebook complete.
